# src02 — Minería de Reglas de Asociación Difusas

**Entrada:** CSV fuzzificado generado por `src01` → `data/<nombre_dataset>_<metrica>_fuzzy.csv`  
**Salida:** CSV de reglas → `data/<nombre_dataset>_<metrica>_reglas.csv`  

Parámetros configurables: nombre_dataset, métrica, soporte mínimo, confianza mínima, lift mínimo, profundidad del beam.


In [ ]:
# @title 1. Imports y configuración

USUARIO = "IZARO"  # @param ["IZARO","ENRIQUE"]
import os
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive

rutas = {
    "IZARO":    "/content/drive/MyDrive/05. PFG/2026 - TFG Izaro Fuzhi/Codigo",
    "ENRIQUE":  "/content/drive/MyDrive/2-DOCENCIA/ALL Trabajos Fin de Estudios/2026 - TFG Izaro Fuzhi/Codigo",
}
DIRECTORIO = rutas[USUARIO]
drive.mount("/content/drive")
os.chdir(DIRECTORIO)
print(f"Directorio de trabajo: {DIRECTORIO}")


In [ ]:
# @title 2. Parámetros del análisis
#SENSOR          = "6823"        # @param ["3600","3730","4301","6791","6822","6823"]
#METRICA         = "ocupacion"  # @param ["intensidad","ocupacion"]

NOMBRE_DATASET = "DailyDelhiClimateTrain"
METRICA = "meantemp"
FICHERO_FUZZY   = f"data/{NOMBRE_DATASET}_{METRICA}_fuzzy.csv"
FICHERO_REGLAS  = f"data/{NOMBRE_DATASET}_{METRICA}_reglas.csv"

# ── Parámetros del análisis ───────────────────────────────────────────────────
_SOPORTE_LABEL    = "Estándar (0.5%)"      # @param ["Muy permisivo (0.1%)", "Permisivo (0.2%)", "Estándar (0.5%)", "Restrictivo (1%)", "Muy restrictivo (2%)"]
_CONFIANZA_LABEL  = "Moderada (50%)"       # @param ["Permisiva (40%)", "Moderada (50%)", "Alta (65%)", "Muy alta (80%)"]
_PROFUNDIDAD_LABEL = "Estándar (3 vars)"   # @param ["Simple (1 var)", "Normal (2 vars)", "Estándar (3 vars)", "Compleja (4 vars)"]
_BEAM_LABEL        = "Estándar (K=10)"     # @param ["Estrecho (K=5)", "Estándar (K=10)", "Amplio (K=20)"]
_LIFT_LABEL = "Sorprendentes" # @param ["Incluir todas","Algo sorprendentes","Sorprendentes","Muy sorprendentes"]

_MAP_SOPORTE   = {"Muy permisivo (0.1%)": 0.001, "Permisivo (0.2%)": 0.002, "Estándar (0.5%)": 0.005, "Restrictivo (1%)": 0.01, "Muy restrictivo (2%)": 0.02}
_MAP_CONFIANZA = {"Permisiva (40%)": 0.40, "Moderada (50%)": 0.50, "Alta (65%)": 0.65, "Muy alta (80%)": 0.80}
_MAP_PROF      = {"Simple (1 var)": 1, "Normal (2 vars)": 2, "Estándar (3 vars)": 3, "Compleja (4 vars)": 4}
_MAP_BEAM      = {"Estrecho (K=5)": 5, "Estándar (K=10)": 10, "Amplio (K=20)": 20}
_MAP_LIFT = {
    "Incluir todas": 1.0,
    "Algo sorprendentes": 1.5,
    "Sorprendentes": 2.0,
    "Muy sorprendentes": 3.0
}
MIN_SOPORTE   = _MAP_SOPORTE[_SOPORTE_LABEL]
MIN_CONFIANZA = _MAP_CONFIANZA[_CONFIANZA_LABEL]
MAX_PROF      = _MAP_PROF[_PROFUNDIDAD_LABEL]
K_BEAM        = _MAP_BEAM[_BEAM_LABEL]
_LIFT_MINIMO = _MAP_LIFT[_LIFT_LABEL]

MAX_PROF        = 3      # @param {"type":"integer"} #Profundidad maxima del beam (nº max de vars en el antecedente)
K_BEAM          = 10     # @param {"type":"integer"} # Anchura del beam (nº de candidatos que se expanden en cada nivel)

# Tolerancia para el filtro de redundancias (ajustar por sensor)
TOP_POR_CONSECUENTE = 10  # @param {"type":"integer"}


# Carga del CSV fuzzificado
df = pd.read_csv(FICHERO_FUZZY)
print(f"Cargado: {FICHERO_FUZZY}")
print(f"Shape: {df.shape[0]:,} filas x {df.shape[1]} columnas")

# Interpretación dinámica según el dataset cargado (se calcula tras cargar df)
_n = len(df)
print(f"─── Parámetros activos ───────────────────────────────")
print(f"  Soporte mínimo : {MIN_SOPORTE} ({MIN_SOPORTE*100:.1f}% → ~{int(MIN_SOPORTE * _n)} filas de {_n:,} en este dataset)")
print(f"  Confianza mín. : {MIN_CONFIANZA}")
print(f"  Profundidad    : {MAX_PROF}")
print(f"  Beam width     : {K_BEAM}")


# Variables antecedentes (temporales) y consecuentes (métricas)
vars_t = [c for c in df.columns if c.startswith("t_")]
vars_v = [c for c in df.columns
          if c.startswith("v_")
          and not c.startswith("v_abs_")
          and c != "v_Mediana"]

print(f"Variables temporales (antecedentes): {len(vars_t)}")
print(f"Variables de métrica  (consecuentes): {len(vars_v)}")
print(f"Consecuentes: {vars_v}")

# Años dinámicos: detectar t_20XX en lugar de hardcodear t_2024/t_2025
_anios_cols = [c for c in df.columns if c.startswith("t_20")]
if _anios_cols:
    print(f"Años detectados en el CSV: {_anios_cols}")


## 3. Funciones (soporte, confianza, lift)

In [ ]:
def calcular_soporte(df, columnas):
    """
    Soporte difuso = suma(min(mu1, mu2, ...)) / N
    Acepta str (1 columna) o lista (intersección AND difusa).
    """
    if isinstance(columnas, str):
        mu = df[columnas]
    else:
        mu = df[list(columnas)].min(axis=1)
    return float(mu.sum() / len(df))


def calcular_confianza(df, antecedente, consecuente):

    ant = [antecedente] if isinstance(antecedente, str) else list(antecedente)
    sop_a = calcular_soporte(df, ant)
    if sop_a == 0:
        return 0.0
    # Soporte conjunto: min de mu_antecedente y mu_consecuente
    mu_ant = df[ant].min(axis=1)          # t-norma del antecedente
    mu_con = df[consecuente]              # pertenencia del consecuente
    sop_ac = float(np.minimum(mu_ant, mu_con).sum() / len(df))
    return float(sop_ac / sop_a)


def calcular_lift(df, antecedente, consecuente):
    """
    Lift = Confianza(A->C) / Soporte(C)
    Lift > 1  -> A y C aparecen juntos más de lo esperado por azar.
    Lift = 1  -> independientes (regla trivial).
    Lift < 1  -> A inhibe C.
    """
    sop_c = calcular_soporte(df, consecuente)
    if sop_c == 0:
        return 0.0
    return float(calcular_confianza(df, antecedente, consecuente) / sop_c)


def evaluar_regla(df, antecedente, consecuente):
    """Devuelve un dict con todas las métricas de una regla."""
    ant = [antecedente] if isinstance(antecedente, str) else list(antecedente)
    return {
        "antecedente": " AND ".join(ant),
        "consecuente": consecuente,
        "n_vars":      len(ant),
        "soporte":     round(calcular_soporte(df, ant),                4),
        "confianza":   round(calcular_confianza(df, ant, consecuente),  4),
        "lift":        round(calcular_lift(df, ant, consecuente),       4),
    }

def calcular_aportacion(df, columnas_antecedente, consecuente, cobertura_acumulada):
  """
  Mide cuánto soporte NUEVO aporta una regla sobre lo ya cubierto.
  cobertura_acumulada es un array con el máximo grado de pertenencia
  ya acumulado por las reglas aceptadas anteriormente.
  """
  mu_ant = df[list(columnas_antecedente)].min(axis=1).to_numpy()
  mu_con = df[consecuente].to_numpy()
  mu_regla = np.minimum(mu_ant, mu_con)
  aportacion = np.maximum(0, mu_regla - cobertura_acumulada)
  return float(aportacion.sum() / len(df))

MESES_POR_ESTACION = {
    "t_Invierno":  {"t_Dic", "t_Ene", "t_Feb"},
    "t_Primavera": {"t_Marz", "t_Abr", "t_May"},
    "t_Verano":    {"t_Jun", "t_Jul", "t_Ago"},
    "t_Otonio":    {"t_Sep", "t_Oct", "t_Nov"},
}

# Grupos mutuamente excluyentes — solo puede aparecer 1 de cada grupo
# Se construyen dinámicamente a partir de las columnas reales del CSV,
# así el código funciona aunque src01 haya desactivado algún bloque.
def _construir_grupos(cols_disponibles):
    """
    Genera los grupos excluyentes filtrando solo las columnas
    que realmente existen en el CSV fuzzificado.
    """
    cols = set(cols_disponibles)
    grupos = []

    candidatos = [
        # Meses
        {"t_Ene","t_Feb","t_Marz","t_Abr","t_May","t_Jun",
         "t_Jul","t_Ago","t_Sep","t_Oct","t_Nov","t_Dic"},
        # Estaciones
        {"t_Primavera","t_Verano","t_Otonio","t_Invierno"},
        # Horas
        {f"t_H{h:02d}" for h in range(24)},
        # Franjas horarias
        {"t_Madrugada","t_Mañana","t_Tarde","t_Noche"},
        # Días de la semana
        {"t_Lun","t_Mar","t_Mie","t_Jue","t_Vie","t_Sab","t_Dom"},
        # Tipo de día
        {"t_Laborable","t_FinSemana"},
        # Años (dinámico: cualquier t_20XX)
        {c for c in cols if c.startswith("t_20")},
        # Quincenas
        {"t_Q1mes","t_Q2mes"},
        # Minutos (cuartos de hora) — nuevo bloque src01
        {"t_M00","t_M15","t_M30","t_M45"},
        # Exclusiones cruzadas laborable/fin de semana
        {"t_Laborable", "t_Sab", "t_Dom"},
        {"t_FinSemana", "t_Lun", "t_Mar", "t_Mie", "t_Jue", "t_Vie"},
        # Exclusiones cruzadas festivo/laborable
        {"t_Laborable", "t_Festivo"},
        {"t_FinSemana", "t_Festivo"},
    ]

    for grupo in candidatos:
        presente = grupo & cols
        if len(presente) >= 2:   # solo añadir si hay al menos 2 vars del grupo
            grupos.append(presente)

    return grupos

GRUPOS_EXCLUYENTES = _construir_grupos(df.columns)
print(f"Grupos excluyentes construidos: {len(GRUPOS_EXCLUYENTES)}")
for g in GRUPOS_EXCLUYENTES:
    print(f"  {sorted(g)}")

def _combinacion_valida(tokens):
    # Regla 1: dentro de cada grupo, máximo 1 valor
    for grupo in GRUPOS_EXCLUYENTES:
        if len(tokens & grupo) > 1:
            return False

    # Regla 2: mes y estación deben ser compatibles
    for estacion, meses_validos in MESES_POR_ESTACION.items():
        if estacion in tokens:
            meses_en_regla = tokens & GRUPOS_EXCLUYENTES[0]
            if meses_en_regla - meses_validos:
                return False
    # Regla 3: hora y franja deben ser compatibles
    HORAS_POR_FRANJA = {
        "t_Madrugada": {f"t_H{h:02d}" for h in range(0, 7)},
        "t_Mañana":    {f"t_H{h:02d}" for h in range(7, 14)},
        "t_Tarde":     {f"t_H{h:02d}" for h in range(14, 21)},
        "t_Noche":     {f"t_H{h:02d}" for h in range(21, 24)},
    }

    for franja, horas_validas in HORAS_POR_FRANJA.items():
        if franja in tokens:
            horas_en_regla = tokens & {f"t_H{h:02d}" for h in range(24)}
            if horas_en_regla and not (horas_en_regla <= horas_validas):
                return False

    # Regla 4: día concreto y tipo_dia deben ser coherentes
    DIAS_LABORABLES = {"t_Lun","t_Mar","t_Mie","t_Jue","t_Vie"}
    DIAS_FINSEM     = {"t_Sab","t_Dom"}
    if "t_Laborable" in tokens and (tokens & DIAS_FINSEM):
        return False
    if "t_FinSemana" in tokens and (tokens & DIAS_LABORABLES):
        return False

    return True


print("Funciones de cálculo definidas: calcular_soporte, calcular_confianza, calcular_lift, evaluar_regla")


## 4. Función beam_search_reglas

In [ ]:
def beam_search_reglas(df, vars_antecedente, consecuente,
                        min_soporte, min_confianza,
                        max_profundidad, k_beam,
                        lift_minimo=1.0):
    """
    Genera reglas difusas SI antecedente -> consecuente mediante beam search.
    Explora combinaciones de variables temporales de profundidad 1 a max_profundidad.
    Acepta una regla solo si su aportación marginal al soporte global supera min_soporte.
    Devuelve DataFrame ordenado por lift DESC.
    """
    reglas_validas = []
    vistos = set()
    beam_actual = [(v,) for v in vars_antecedente]
    cobertura_acumulada = np.zeros(len(df))

    for profundidad in range(1, max_profundidad + 1):
        print(f"  Profundidad {profundidad}: {len(beam_actual):,} candidatos")
        puntuaciones = []

        for candidato in beam_actual:
            clave = tuple(sorted(candidato))
            if clave in vistos:
                continue
            vistos.add(clave)

            # Filtro semántico: descartar mes+estación incompatibles
            if not _combinacion_valida(set(clave)):
                continue

            # Filtro de soporte mínimo
            sop = calcular_soporte(df, list(clave))
            if sop < min_soporte:
                continue

            conf = calcular_confianza(df, list(clave), consecuente)
            lift = calcular_lift(df, list(clave), consecuente)

            if conf >= min_confianza and lift >= lift_minimo:
                aportacion = calcular_aportacion(
                    df, list(clave), consecuente, cobertura_acumulada)
                if aportacion >= min_soporte:
                    reglas_validas.append(evaluar_regla(df, list(clave), consecuente))
                    mu_ant = df[list(clave)].min(axis=1).to_numpy()
                    mu_con = df[consecuente].to_numpy()
                    cobertura_acumulada = np.maximum(
                        cobertura_acumulada,
                        np.minimum(mu_ant, mu_con)
                    )

            puntuaciones.append((clave, conf))

        if not puntuaciones or profundidad == max_profundidad:
            break

        top_k = sorted(puntuaciones, key=lambda x: x[1], reverse=True)[:k_beam]

        beam_siguiente = []
        for clave, _ in top_k:
            vars_disponibles = [v for v in vars_antecedente if v not in clave]
            for nueva_var in vars_disponibles:
                nuevo = tuple(sorted(set(clave) | {nueva_var}))
                if nuevo not in vistos:
                    beam_siguiente.append(nuevo)

        beam_actual = list(dict.fromkeys(beam_siguiente))
        if not beam_actual:
            break

    if not reglas_validas:
        return pd.DataFrame(columns=["antecedente","consecuente","n_vars",
                                      "soporte","confianza","lift"])

    return (pd.DataFrame(reglas_validas)
              .drop_duplicates(subset=["antecedente","consecuente"])
              .sort_values("lift", ascending=False)
              .reset_index(drop=True))

## 5. Funciones de filtrado

### *Filtro 1: redundancias por subconjunto*
Elimina una regla si existe otra con un subconjunto estricto de sus variables (mismo consecuente) cuyo lift sea similar o mejor. Ajusta `TOLERANCIA_LIFT` en la celda 2 según el sensor.

In [ ]:
def filtrar_redundantes(df_reglas, min_confianza):
    """
    Entre dos reglas donde el antecedente de una es subconjunto del otro
    (mismo consecuente), conserva la más corta siempre que su confianza
    supere min_confianza. El lift se usa solo para ordenar, no para filtrar.
    """
    registros = df_reglas.to_dict("records")
    mantener = []

    for i, fila in enumerate(registros):
        vars_fila = set(fila["antecedente"].split(" AND "))
        es_redundante = False

        for j, otra in enumerate(registros):
            if i == j:
                continue
            vars_otra = set(otra["antecedente"].split(" AND "))
            if (vars_otra < vars_fila                        # otra es subconjunto estricto
                    and otra["consecuente"] == fila["consecuente"]
                    and otra["confianza"] >= min_confianza): # la más corta supera el mínimo
                es_redundante = True
                break

        if not es_redundante:
            mantener.append(fila)

    return pd.DataFrame(mantener).reset_index(drop=True)

### *Filtro 2: redundancias jerárquicas*

Si existe una regla con variable padre (ej: `t_Tarde`) y otra equivalente con variable hijo (ej: `t_H14`), elimina la del hijo.

In [ ]:
def filtrar_por_jerarquia(df_reglas, jerarquia, min_confianza):
    """
    Elimina la regla con variable "hijo" si existe la regla equivalente
    con la variable "padre" y su confianza supera min_confianza.
    """
    registros = df_reglas.to_dict("records")
    mantener = []

    for i, fila in enumerate(registros):
        vars_fila = set(fila["antecedente"].split(" AND "))
        es_redundante = False

        for padre, hijos in jerarquia.items():
            hijos_en_fila = vars_fila & set(hijos)
            if not hijos_en_fila:
                continue
            # Construir cómo sería la regla equivalente con el padre
            vars_con_padre = (vars_fila - hijos_en_fila) | {padre}

            for j, otra in enumerate(registros):
                if i == j:
                    continue
                vars_otra = set(otra["antecedente"].split(" AND "))
                if (vars_otra == vars_con_padre
                        and otra["consecuente"] == fila["consecuente"]
                        and otra["confianza"] >= min_confianza):
                    es_redundante = True
                    break
            if es_redundante:
                break

        if not es_redundante:
            mantener.append(fila)

    return pd.DataFrame(mantener).reset_index(drop=True)

### *Filtro 3: límite por consecuente*
Limita el número máximo de reglas por consecuente para evitar saturación en src03.

In [ ]:
def filtrar_top_por_consecuente(df_reglas, top_n):
    """
    Limita el número máximo de reglas por consecuente
    para evitar saturación en src03.
    """
    return (df_reglas
            .sort_values("lift", ascending=False)
            .groupby("consecuente", group_keys=False)
            .head(top_n)
            .sort_values("lift", ascending=False)
            .reset_index(drop=True))

print("filtrar_top_por_consecuente definida.")

## 6. Diagnóstico de umbrales

(OPCIONAL)
> **Solo ejecutar cuando se cambia de sensor y no se sabe qué umbrales usar.**  
> No forma parte del pipeline principal. El resultado orienta qué valores poner en MIN_CONFIANZA y MIN_LIFT (punto 2).

In [ ]:
# 🔴 ¿Por qué no evaluar con el soporte? --> El barrido de parámetros se centra en la confianza y el lift para optimizar la densidad de información del resumen.
# ──────────────────────────────────────────────────────────────────

# HERRAMIENTA DE DIAGNÓSTICO — no forma parte del pipeline principal.
# ──────────────────────────────────────────────────────────────────
# Usa umbrales mínimos intencionalmente para explorar el espacio de reglas
# antes de fijar MIN_CONFIANZA. No ejecutar en producción.
# el lift se usa para ORDENAR, no para FILTRAR.
# El barrido ahora es 1D: solo varía MIN_CONFIANZA.
# El resultado ya sale ordenado por lift DESC — las reglas más fuertes primero.
# ──────────────────────────────────────────────────────────────────

# Paso 1: beam search con umbrales mínimos
print("Ejecutando beam search base...")
todos_raw = []
for consecuente in vars_v:
    print(f"  Consecuente: {consecuente}")
    df_r = beam_search_reglas(
        df,
        vars_antecedente = vars_t,
        consecuente      = consecuente,
        min_soporte      = 0.01,
        min_confianza    = 0.30,
        max_profundidad  = MAX_PROF,
        k_beam           = K_BEAM,
        lift_minimo      = 1, #lift_minimo = 1.0  # mínimo absoluto para diagnóstico

    )
    if not df_r.empty:
        todos_raw.append(df_r)

df_raw = (pd.concat(todos_raw, ignore_index=True) if todos_raw
          else pd.DataFrame(columns=["antecedente","consecuente","n_vars",
                                     "soporte","confianza","lift"]))
print(f"Reglas crudas totales: {len(df_raw)}")

# Paso 2: barrido 1D — solo confianza (lift ya no filtra, ordena)
N_IDEAL_MIN = 15
N_IDEAL_MAX = 40

confianzas = [0.40, 0.42, 0.45, 0.48, 0.50, 0.52, 0.55,
              0.58, 0.60, 0.65, 0.70, 0.75, 0.80]

print(f"\n{'confianza':>10} {'reglas':>8} {'lift_medio':>12} {'lift_max':>10}")
print("-" * 44)
for min_conf in confianzas:
    sub = df_raw[df_raw["confianza"] >= min_conf]
    n = len(sub)
    marca = " ✓" if N_IDEAL_MIN <= n <= N_IDEAL_MAX else ""
    lift_med = sub["lift"].mean() if n > 0 else 0.0
    lift_max = sub["lift"].max()  if n > 0 else 0.0
    print(f"{min_conf:>10.2f} {n:>8} {lift_med:>12.2f} {lift_max:>10.2f}{marca}")

print("\n→ Elige el MIN_CONFIANZA marcado con ✓ y ponlo en la celda 2.")
print("  Las reglas saldrán automáticamente ordenadas por lift (las más fuertes primero).")

## 7. Jerarquía semántica de variables temporales

In [ ]:
# Jerarquía padre → hijos (robusta a columnas ausentes)
# Si existe una regla con la variable "padre" (ej: t_Tarde),
# el filtro jerárquico elimina la regla equivalente con la variable "hijo" (ej: t_H14)
# siempre que el lift sea similar.

_cols_csv = set(df.columns)

def _filtrar_hijos(hijos):
    """Devuelve solo los hijos que existen en el CSV."""
    return [h for h in hijos if h in _cols_csv]

_JERARQUIA_COMPLETA = {
    # Franjas → horas individuales
    "t_Madrugada": [f"t_H{h:02d}" for h in range(0,  7)],
    "t_Mañana":    [f"t_H{h:02d}" for h in range(7,  14)],
    "t_Tarde":     [f"t_H{h:02d}" for h in range(14, 21)],
    "t_Noche":     [f"t_H{h:02d}" for h in range(21, 24)],
    # Tipo día → días individuales
    "t_Laborable": ["t_Lun", "t_Mar", "t_Mie", "t_Jue", "t_Vie"],
    "t_FinSemana": ["t_Sab", "t_Dom"],
    # Estaciones → meses
    "t_Invierno":  ["t_Dic", "t_Ene", "t_Feb"],
    "t_Primavera": ["t_Marz", "t_Abr", "t_May"],
    "t_Verano":    ["t_Jun", "t_Jul", "t_Ago"],
    "t_Otonio":    ["t_Sep", "t_Oct", "t_Nov"],
    # Festivo → laborable/fin de semana (festivo es más específico)
    "t_Festivo":   ["t_Laborable", "t_FinSemana"],
}

JERARQUIA = {
    padre: hijos_filtrados
    for padre, hijos in _JERARQUIA_COMPLETA.items()
    if padre in _cols_csv
    for hijos_filtrados in [_filtrar_hijos(hijos)]
    if hijos_filtrados  # solo incluir si hay al menos 1 hijo presente
}

print(f"JERARQUIA activa ({len(JERARQUIA)} entradas, filtrada según columnas del CSV):")
for padre, hijos in JERARQUIA.items():
    print(f"  {padre}: {hijos}")


## 8. Beam search principal

In [ ]:
# @title
todos = []

for consecuente in vars_v:
    sep = "=" * 55
    print(f"\n{sep}")
    print(f"Consecuente: {consecuente}")
    print(sep)

    df_r = beam_search_reglas(
        df,
        vars_antecedente = vars_t,
        consecuente      = consecuente,
        min_soporte      = MIN_SOPORTE,
        min_confianza    = MIN_CONFIANZA,
        max_profundidad  = MAX_PROF,
        k_beam           = K_BEAM,
        lift_minimo      = _LIFT_MINIMO,
    )

    if not df_r.empty:
        print(f"  -> {len(df_r)} reglas encontradas")
        print(df_r.to_string(index=False))
        todos.append(df_r)
    else:
        print("  Sin reglas para este consecuente.")

if todos:
    df_reglas = (pd.concat(todos, ignore_index=True)
                   .sort_values("lift", ascending=False)
                   .reset_index(drop=True))
else:
    df_reglas = pd.DataFrame(columns=["antecedente","consecuente","n_vars",
                                       "soporte","confianza","lift"])



print(f"\nTotal reglas antes de filtrar redundancias: {len(df_reglas)}")




## 9. Llamadas a los filtros

In [ ]:
if df_reglas.empty:
    print("Sin reglas que filtrar.")
else:
    df_reglas = filtrar_redundantes(df_reglas, min_confianza=MIN_CONFIANZA)
    print(f"Reglas tras filtro 1 (redundancias por subconjunto): {len(df_reglas)}")
    df_reglas

    df_reglas = filtrar_por_jerarquia(df_reglas, JERARQUIA, min_confianza=MIN_CONFIANZA)
    print(f"Reglas tras filtro 2 (jerarquía semántica): {len(df_reglas)}")
    df_reglas

    df_reglas = filtrar_top_por_consecuente(df_reglas, TOP_POR_CONSECUENTE)
    print(f"Reglas tras filtro 3 (top {TOP_POR_CONSECUENTE} por consecuente): {len(df_reglas)}")



## 10. Guardado a CSV y resumen

In [ ]:
if df_reglas.empty:
    print("Sin reglas para guardar. Revisa los umbrales en la celda 2.")
else:
    df_reglas.to_csv(FICHERO_REGLAS, index=False)
    print(f"Reglas guardadas en: {FICHERO_REGLAS}")
    print(f"Shape: {df_reglas.shape}")

    print("\nResumen por consecuente:")
    resumen = (df_reglas
               .groupby("consecuente")
               .agg(
                   n_reglas   = ("lift", "count"),
                   conf_media = ("confianza", "mean"),
                   lift_medio = ("lift", "mean"),
                   lift_max   = ("lift", "max"),
               )
               .round(4)
               .sort_values("lift_medio", ascending=False))
    print(resumen.to_string())


## 11. Visualización

In [ ]:
if df_reglas.empty:
    print("Sin reglas para visualizar.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Scatter soporte vs confianza, color = lift
    ax = axes[0]
    sc = ax.scatter(
        df_reglas["soporte"], df_reglas["confianza"],
        c=df_reglas["lift"], cmap="YlOrRd",
        alpha=0.75, edgecolors="gray", linewidths=0.4
    )
    plt.colorbar(sc, ax=ax, label="Lift")
    ax.axhline(MIN_CONFIANZA, color="steelblue", linestyle="--",
               linewidth=0.8, label=f"conf mín={MIN_CONFIANZA}")
    ax.axvline(MIN_SOPORTE, color="firebrick", linestyle="--",
               linewidth=0.8, label=f"sop mín={MIN_SOPORTE}")
    ax.set_xlabel("Soporte")
    ax.set_ylabel("Confianza")
    ax.set_title("Soporte vs Confianza  (color = Lift)")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

    # Top 15 reglas por lift
    ax = axes[1]
    top15 = df_reglas.head(15).copy()
    etiquetas = [f"{r.antecedente}\n-> {r.consecuente}" for _, r in top15.iterrows()]
    rango = top15["lift"].max() - top15["lift"].min() + 1e-9
    colores = plt.cm.YlOrRd((top15["lift"] - top15["lift"].min()) / rango)
    ax.barh(range(len(top15)), top15["lift"],
            color=colores, edgecolor="gray", linewidth=0.4)
    ax.set_yticks(range(len(top15)))
    ax.set_yticklabels(etiquetas, fontsize=7)
    ax.invert_yaxis()
    ax.set_xlabel("Lift")
    ax.set_title("Top 15 reglas por Lift")
    ax.axvline(_LIFT_MINIMO, color="steelblue", linestyle="--", linewidth=0.8,
               label=f"lift mín={_LIFT_MINIMO}")
    ax.legend(fontsize=8)
    ax.grid(axis="x", alpha=0.3)

    plt.suptitle(f"{NOMBRE_DATASET}", fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()

In [ ]:
!cp data/{NOMBRE_DATASET}_{METRICA}_reglas.csv data/{NOMBRE_DATASET}_{METRICA}_reglas_ANTES.csv

## 12. Diff antes/después del cambio de lift
Compara el CSV de reglas guardado antes del cambio (baseline)
con el recién generado. Ejecutar solo una vez, justo después
de haber regenerado las reglas con el nuevo `LIFT_MINIMO_ACEPTABLE`.


In [ ]:
# ── Diff de reglas: antes vs después del cambio de lift ─────────────────────
# Instrucciones:
#   1. Antes de aplicar el cambio, guarda una copia del CSV actual:
#      !cp data/{SENSOR}_{METRICA}_reglas.csv data/{SENSOR}_{METRICA}_reglas_ANTES.csv
#   2. Aplica los cambios de lift y ejecuta el beam (celda 19 + celda 21-23).
#   3. Ejecuta esta celda para ver exactamente qué cambió.
# ────────────────────────────────────────────────────────────────────────────

RUTA_ANTES   = f"data/{NOMBRE_DATASET}_{METRICA}_reglas_ANTES.csv"
RUTA_DESPUES = f"data/{NOMBRE_DATASET}_{METRICA}_reglas.csv"

try:
    df_antes   = pd.read_csv(RUTA_ANTES)
    df_despues = pd.read_csv(RUTA_DESPUES)
except FileNotFoundError as e:
    print(f"⚠️  {e}")
    print("   Guarda primero la copia del CSV original (ver instrucciones arriba).")
    raise

# Construir claves únicas
df_antes["_clave"]   = df_antes["antecedente"]   + " → " + df_antes["consecuente"]
df_despues["_clave"] = df_despues["antecedente"] + " → " + df_despues["consecuente"]

desaparecidas = df_antes[~df_antes["_clave"].isin(df_despues["_clave"])].copy()
nuevas        = df_despues[~df_despues["_clave"].isin(df_antes["_clave"])].copy()

# Reglas que permanecen pero cambiaron métricas
claves_comunes = set(df_antes["_clave"]) & set(df_despues["_clave"])
cambios_metrica = []
for clave in claves_comunes:
    a = df_antes[df_antes["_clave"] == clave].iloc[0]
    d = df_despues[df_despues["_clave"] == clave].iloc[0]
    if abs(a["lift"] - d["lift"]) > 0.01 or abs(a["confianza"] - d["confianza"]) > 0.005:
        cambios_metrica.append({
            "regla":        clave,
            "lift_antes":   a["lift"],
            "lift_despues": d["lift"],
            "conf_antes":   a["confianza"],
            "conf_despues": d["confianza"],
        })

# ── Resumen ──────────────────────────────────────────────────────────────────
print("=" * 60)
print(f"DIFF DE REGLAS —  {FICHERO_FUZZY}")
print("=" * 60)
print(f"  Reglas antes:        {len(df_antes)}")
print(f"  Reglas después:      {len(df_despues)}")
print(f"  Δ total:             {len(df_despues) - len(df_antes):+d}")
print(f"  Desaparecidas:       {len(desaparecidas)}")
print(f"  Nuevas:              {len(nuevas)}")
print(f"  Con métricas camb.:  {len(cambios_metrica)}")

# ── Reglas que desaparecieron ─────────────────────────────────────────────
if not desaparecidas.empty:
    print(f"\n── Reglas que DESAPARECIERON ({len(desaparecidas)}) ──")
    print(desaparecidas[["antecedente","consecuente","confianza","lift"]]
          .sort_values("lift", ascending=False)
          .to_string(index=False))
else:
    print("\n── Sin reglas que desaparecieron ✅")

# ── Reglas nuevas ────────────────────────────────────────────────────────────
if not nuevas.empty:
    print(f"\n── Reglas NUEVAS ({len(nuevas)}) — ordenadas por lift ──")
    print(nuevas[["antecedente","consecuente","confianza","lift"]]
          .sort_values("lift", ascending=False)
          .to_string(index=False))
    print(f"\n   Lift medio de reglas nuevas: {nuevas['lift'].mean():.2f}")
    print(f"   Lift mín de reglas nuevas:   {nuevas['lift'].min():.2f}")
    print(f"   Confianza media nuevas:       {nuevas['confianza'].mean()*100:.1f}%")
else:
    print("\n── Sin reglas nuevas")

# ── Reglas con métricas cambiadas ────────────────────────────────────────────
if cambios_metrica:
    print(f"\n── Reglas con MÉTRICAS MODIFICADAS ({len(cambios_metrica)}) ──")
    df_cambios = pd.DataFrame(cambios_metrica).round(3)
    print(df_cambios.to_string(index=False))

# ── Limpieza de columna auxiliar ─────────────────────────────────────────────
df_antes.drop(columns=["_clave"], inplace=True)
df_despues.drop(columns=["_clave"], inplace=True)